In [ ]:
!export LDFLAGS="-L/opt/homebrew/opt/libffi/lib"

In [ ]:
!export CPPFLAGS="-I/opt/homebrew/opt/libffi/include"

In [2]:
import sqlite3
from jinja2 import Environment, FileSystemLoader
from weasyprint import HTML
from datetime import date
import smtplib
from email.message import EmailMessage
import os
import re

#import sqlite3

conn = sqlite3.connect("invoices.db")
c = conn.cursor()

c.execute("""
CREATE TABLE IF NOT EXISTS invoices (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    invoice_number TEXT UNIQUE
)
""")
conn.commit()



In [3]:
def get_next_invoice_number():
        conn = sqlite3.connect("invoices.db")
        c = conn.cursor()

        c.execute("SELECT COUNT(*) FROM invoices")
        count = c.fetchone()[0] + 1
        invoice_number = f"INV-PRO-2026-{count:04d}"

        c.execute(
            "INSERT INTO invoices (invoice_number) VALUES (?)",
            (invoice_number,))

        conn.commit()
        conn.close()

        return invoice_number

        #global numb

In [ ]:
def get_facture_pro(data):
    
    def generate_invoice_pdf(data, output_file):
        env = Environment(loader=FileSystemLoader('.'))
        template = env.get_template('base.html')

        html_content = template.render(**data)

        HTML(string=html_content, base_url=os.getcwd()).write_pdf(output_file)

   
    
    invoice_data = {
    "invoice_number": data['num'],
    "date": date.today().strftime("%d %b %Y"),
    "client_name": data['name'],
    "client_address": data['addr'],
    "email": data['email'],
    "telephone": data['tel'],
    "objet": data['obj'],
    "items": data['items'],}

    VAT_RATE = 0.16

    subtotal = sum(item["quantity"] * item["unit_price"] for item in invoice_data["items"])
    
    invoice_data["subtotal"] = subtotal

    vat = subtotal * VAT_RATE
    invoice_data["TVA"] = str(VAT_RATE * 100) + "%"
    total = subtotal + vat
    invoice_data["tva"] = vat

    invoice_data["total"] = total

    pdf_file = data['num'] + ".pdf"

    generate_invoice_pdf(invoice_data, pdf_file)
    #email_invoice(pdf_file, data['email'], data['num'])

      
    #print("Invoice generated and emailed successfully.")

In [6]:
launch_invoice_gui()
#get_facture_pro(dico)

In [2]:
# get_facture_pro(dico)